# Phase 10: Voice notes and photos

A report can arrive as a voice note. Speech-to-text (Gemini) turns it into text, and the citizen checks that text before it enters the same pipeline. Photos are optional and only for the human reviewer.

Sections 1 to 4 run on their own. Section 5 needs the API: open a new terminal in VS Code, run the command below, and leave it running.

```bash
uv run uvicorn community_voice.api.main:app --port 8800
```

In [ ]:
import io
import re

import httpx
from PIL import Image

from community_voice.agents.triage import triage
from community_voice.core.config import PROJECT_ROOT
from community_voice.media.photos import InvalidPhoto, clean_photo
from community_voice.media.transcribe import transcribe
from community_voice.schemas.report import ReportIn

AUDIO = PROJECT_ROOT / "data" / "sample_reports" / "audio"

## 1. One voice note

In [ ]:
path = AUDIO / "English.mp3"
transcript = transcribe(path.read_bytes(), path.name)
transcript

## 2. Every language

Gemini is not told the language. Every transcript comes with a warning, so the citizen always checks the text.

In [ ]:
for path in sorted(AUDIO.glob("*.mp3")):
    t = transcribe(path.read_bytes(), path.name)
    print(f"{path.stem}:")
    print("  ", t.text[:150])
print(t.warning)

## 3. Into the same pipeline

After the citizen confirms the text, it is an ordinary report.

In [ ]:
report = ReportIn(text=transcript.text)
result = triage(report)
print(result.category.value, result.urgency.value)
print(result.summary)

## 4. Photos

Phone photos carry hidden data, including GPS location. We build a photo with fake metadata, then clean it: the image is decoded to prove it is real, resized, and re-saved without the metadata.

In [ ]:
exif = Image.Exif()
exif[0x010F] = "TestPhone"
exif.get_ifd(0x8825)[1] = "S"  # a GPS entry

buffer = io.BytesIO()
Image.new("RGB", (3200, 2400), "grey").save(buffer, "JPEG", exif=exif.tobytes())
original = buffer.getvalue()
cleaned = clean_photo(original)


def describe(data):
    image = Image.open(io.BytesIO(data))
    return (
        f"{image.size}, {len(image.getexif())} metadata entries, {len(data) // 1024} KB"
    )


print("before:", describe(original))
print("after: ", describe(cleaned))

In [ ]:
try:
    clean_photo(b"this is not a photo")
except InvalidPhoto as error:
    print(error)

## 5. Through the API

In [ ]:
BASE = "http://127.0.0.1:8800"

with (AUDIO / "English.mp3").open("rb") as file:
    response = httpx.post(
        BASE + "/transcriptions", files={"audio": ("English.mp3", file)}, timeout=60
    )
voice = response.json()
voice

In [ ]:
response = httpx.post(BASE + "/reports", json={"text": voice["text"]}, timeout=180)
report_id = re.search(r"r-\w+", response.text).group()
report_id

In [ ]:
files = {"photo": ("pothole.jpg", original, "image/jpeg")}
print(httpx.put(f"{BASE}/reports/{report_id}/photo", files=files).json())
print(httpx.get(f"{BASE}/reports/{report_id}").json()["has_photo"])

In [ ]:
bad = {"photo": ("x.jpg", b"not a photo", "image/jpeg")}
response = httpx.put(f"{BASE}/reports/{report_id}/photo", files=bad)
print(response.status_code, response.json())

## What to notice

- The voice note became text, and the pipeline did not change.
- Every transcript carries a warning, even English.
- The photo is optional, checked by decoding it, and its metadata is gone.